# AIC 2026 — Mô tả keyframes bằng VLM Qwen3.5-0.8B

Sinh **caption tiếng Anh** cho từng keyframe trên Google Drive, lưu JSON theo từng video, có checkpoint để chạy tiếp sau khi Colab ngắt kết nối. Cấu trúc input/output giống hệt notebook OCR nên hai nguồn dữ liệu ghép chung được.

Bản 0.8B nhẹ, chạy được trên T4 free — dùng để quét nhanh toàn bộ dataset. Cần mô tả chi tiết hơn thì dùng notebook `AIC_VLM_Keyframes_Qwen3_5_4B_Colab.ipynb`.

In [ ]:
!nvidia-smi

# Nguyên tắc giống notebook OCR: KHÔNG đụng vào torch/numpy/Pillow/opencv của Colab.
# transformers không phụ thuộc torch nên nâng cấp nó an toàn; accelerate chỉ yêu cầu
# torch>=2.0 (Colab đã thoả) nên pip giữ nguyên torch.
!pip -q install -U transformers accelerate

# Nếu môi trường đã hỏng từ lần chạy trước: Runtime → Disconnect and delete runtime.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cấu hình

Chỉnh cho **16GB VRAM**: fp16, `BATCH_SIZE=16`, ảnh 448px; độ dài caption do PROMPT quyết định (trần 200 token chỉ là lưới an toàn). Model 0.8B chỉ chiếm ~1.6GB nên phần lớn VRAM dành cho batch — OOM thì hạ `BATCH_SIZE` trước, `MAX_IMAGE_SIDE` sau.

Ba số ảnh hưởng tốc độ nhiều nhất: `MAX_IMAGE_SIDE` (chi phí ~ bình phương), độ dài caption yêu cầu trong `PROMPT` (decode tuần tự nên caption dài là tốn) và `DEDUP_MAX_DISTANCE` (bỏ hẳn ảnh trùng). Đổi `PROMPT` sẽ đổi `run_id`, notebook sinh lại caption chứ không nhận nhầm file cũ.

In [ ]:
from pathlib import Path

DATASET_DIRECTORY = Path('/content/drive/MyDrive/AI Challenge/Dataset_Directory')
VLM_DIRECTORY = Path('/content/drive/MyDrive/AI Challenge/VLM_Qwen3.5-0.8B')
MAP_KEYFRAMES_DIRECTORY = DATASET_DIRECTORY / 'map-keyframes-aic25-b1' / 'map-keyframes'

# CHIA NHIỀU LẦN CHẠY: chỉ để lại vài folder ở đây, comment phần còn lại.
# Lần sau đổi sang nhóm khác. Output ghi chung một thư mục Drive, mỗi video một
# file .json riêng nên các lần chạy không đè nhau và không cần gộp thủ công.
TARGET_FOLDERS = [
    'Keyframes_L21', 'Keyframes_L22', 'Keyframes_L23', 'Keyframes_L24',
    'Keyframes_L25', 'Keyframes_L26_a', 'Keyframes_L26_b',
    # --- lần chạy sau: bỏ comment nhóm dưới, comment nhóm trên ---
    'Keyframes_L26_c', 'Keyframes_L26_d', 'Keyframes_L26_e',
    'Keyframes_L27', 'Keyframes_L28', 'Keyframes_L29', 'Keyframes_L30',
]

# --- Model -----------------------------------------------------------------
MODEL_ID = 'Qwen/Qwen3.5-0.8B'
DTYPE = 'float16'              # 'auto' | 'float16' | 'bfloat16'
LOAD_IN_4BIT = False  # bitsandbytes NF4, giảm VRAM ~4x, chậm hơn chút
ATTENTION = 'sdpa'

# --- Sinh caption ----------------------------------------------------------
CAPTION_LANGUAGE = 'en'
# Prompt viết cho SEARCH INDEX, không phải cho người đọc. Ép model nói ra đúng
# những thứ CLIP yếu: đếm số lượng, quan hệ trái/phải/trước/sau, màu quần áo,
# phủ định. Caption "đẹp" mà chung chung thì vô dụng khi fuse với CLIP.
# Thứ tự các mục là CÓ CHỦ ĐÍCH: nếu output bị cắt vì hết token thì phần mất đi
# là phần cuối, nên thứ giá trị nhất cho tìm kiếm phải đứng trước.
PROMPT = (
    'Describe this video keyframe for a search index in at most 4 short English sentences, '
    'under 70 words total. Sentence 1: how many people are visible, what they are doing, '
    'and their clothing colours. Sentence 2: any readable on-screen text, channel logo or '
    'caption bar, quoted exactly. Sentence 3: the main objects and vehicles with their colours, '
    'and where they sit in the frame (left, right, foreground, background). '
    'Sentence 4: the setting, indoors or outdoors, day or night. '
    'Start directly with the content, never with "The image shows". '
    'Be terse and factual: no filler adjectives, no speculation about names, places, '
    'intentions or emotions.'
)
# Độ dài do PROMPT quyết định, không cắt cứng sau khi sinh. Số dưới đây chỉ là
# LƯỚI AN TOÀN chống trường hợp model lặp vô hạn — generate() dừng ngay khi model
# phát EOS, nên đặt rộng gần như không tốn thêm gì. Cell test in tỉ lệ chạm trần;
# nếu ~0% thì trần này không hề can thiệp vào độ dài caption.
MAX_NEW_TOKENS = 200
BATCH_SIZE = 16          # số keyframe sinh caption cùng lúc; giảm nếu OOM
MAX_IMAGE_SIDE = 448  # chi phí ~ BÌNH PHƯƠNG số này (số visual token)

# --- Lọc keyframe trùng ----------------------------------------------------
# Keyframe liên tiếp trong cùng một shot gần như giống hệt nhau. So dHash 64-bit
# với ảnh đại diện của nhóm; frame trùng chép lại caption thay vì gọi model.
# JSON output vẫn đủ mọi keyframe, chỉ thêm trường 'duplicate_of'.
#   0  = chỉ gộp ảnh gần y hệt (an toàn)
#   4  = mặc định, gộp cả thay đổi nhỏ (đèn nhấp nháy, subtitle đổi)
#   8+ = gộp mạnh, có thể mất chi tiết
#   -1 = tắt hẳn, caption mọi keyframe
DEDUP_MAX_DISTANCE = 4
DEDUP_HASH_SIZE = 8

# Ghi checkpoint ra đĩa local mỗi batch (rẻ), đẩy lên Drive mỗi N batch (Drive I/O chậm).
DRIVE_SYNC_EVERY = 10

# --- Chia đợt chạy ---------------------------------------------------------
# Colab hay ngắt session, nên chạy dần từng đợt vài video thay vì một lèo.
#   RUN_SLICE = '0:3'  -> lấy 3 video ĐẦU TIÊN CHƯA XONG. Chạy lại cell là 3 video
#                         tiếp theo, không cần sửa số. Đợt sau muốn 4 thì đổi '0:4'.
#   RUN_SLICE = ''     -> lấy hết phần chưa xong (chạy một lèo).
#   SKIP_DONE = False  -> chỉ số tính trên TOÀN BỘ danh sách: đợt 1 '0:3', đợt 2 '3:7'…
#   RUN_VIDEO_IDS      -> chỉ định thẳng video, ưu tiên hơn RUN_SLICE.
RUN_SLICE = '0:3'
SKIP_DONE = True
RUN_VIDEO_IDS = []  # ví dụ: ['L21_V001', 'L22_V015']

OVERWRITE = False
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}

dataset_root = DATASET_DIRECTORY.resolve()
KEYFRAME_ROOTS, missing = [], []
for folder in TARGET_FOLDERS:
    relative = Path(folder.strip())
    assert not relative.is_absolute(), f'Chỉ nhận đường dẫn tương đối: {folder}'
    selected = (dataset_root / relative).resolve()
    assert selected == dataset_root or dataset_root in selected.parents
    if selected.is_dir(): KEYFRAME_ROOTS.append(selected)
    else: missing.append(folder)
assert KEYFRAME_ROOTS, 'Không tìm thấy thư mục keyframe nào'
assert MAP_KEYFRAMES_DIRECTORY.is_dir(), f'Thiếu map-keyframes: {MAP_KEYFRAMES_DIRECTORY}'
OUTPUT_ROOT = VLM_DIRECTORY.resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if missing: print('Bỏ qua thư mục không tồn tại:', ', '.join(missing))
print('Input:', len(KEYFRAME_ROOTS), '| Output:', OUTPUT_ROOT)

## Quét video + map-keyframes

Giống notebook OCR: mỗi thư mục `L\d\d_V\d\d\d` là một video, `map-keyframes/<video>.csv` cho `frame_idx`/`pts_time` để nối kết quả về timeline.

In [ ]:
import csv
import re

VIDEO_ID_PATTERN = re.compile(r'^L\d{2}_V\d{3}$')

def find_video_dirs(root):
    base = root / 'keyframes'
    if not base.is_dir(): base = root
    return [p for p in sorted(base.iterdir()) if p.is_dir() and VIDEO_ID_PATTERN.match(p.name)]

def load_keyframe_map(video_id):
    path = MAP_KEYFRAMES_DIRECTORY / f'{video_id}.csv'
    if not path.is_file(): return None
    mapping = {}
    with path.open(encoding='utf-8-sig', newline='') as handle:
        for row in csv.DictReader(handle):
            try:
                mapping[int(row['n'])] = {'pts_time': float(row['pts_time']), 'fps': float(row['fps']), 'frame_idx': int(row['frame_idx'])}
            except (KeyError, TypeError, ValueError): pass
    return mapping or None

def keyframe_order(path):
    return int(path.stem) if path.stem.isdigit() else None

video_dirs = sorted((v for root in KEYFRAME_ROOTS for v in find_video_dirs(root)), key=lambda p: p.name)
print(f'Tìm thấy {len(video_dirs)} video; {sum(load_keyframe_map(v.name) is not None for v in video_dirs)} video có map')

## Tải model

Notebook chỉ dùng đúng `Qwen/Qwen3.5-0.8B`, không có model dự phòng. `AutoModelForImageTextToText` là API chung cho VLM của Qwen. Nếu cell này báo lỗi tải (repo không tồn tại / không phải bản vision / cần token), sửa `MODEL_ID` ở cell cấu hình.

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

use_gpu = torch.cuda.is_available()
print('Device:', torch.cuda.get_device_name(0) if use_gpu else 'CPU')
if not use_gpu:
    print('CẢNH BÁO: chạy CPU sẽ rất chậm. Runtime → Change runtime type → GPU.')

torch_dtype = {'auto': 'auto', 'float16': torch.float16, 'bfloat16': torch.bfloat16}[DTYPE]
load_kwargs = {'dtype': torch_dtype, 'device_map': 'auto' if use_gpu else None, 'attn_implementation': ATTENTION}
if LOAD_IN_4BIT:
    from transformers import BitsAndBytesConfig
    load_kwargs['quantization_config'] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
    )
    load_kwargs.pop('dtype', None)

print('Đang tải', MODEL_ID, '...')
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, trust_remote_code=True, **load_kwargs)
model.eval()
if not use_gpu:
    model.to('cpu')
# Batched generation cần padding bên trái, nếu không caption sẽ lệch.
tokenizer = getattr(processor, 'tokenizer', processor)
tokenizer.padding_side = 'left'
if getattr(tokenizer, 'pad_token', None) is None:
    tokenizer.pad_token = tokenizer.eos_token

RUN_ID = f"{MODEL_ID}|caption-{CAPTION_LANGUAGE}"
print('Đã tải:', MODEL_ID, '| dtype:', DTYPE, '| 4bit:', LOAD_IN_4BIT)

## Sinh caption + lọc trùng

Mỗi keyframe đi qua chat template có 1 ảnh + `PROMPT`, sinh theo batch, `do_sample=False` (greedy) để chạy lại cho kết quả ổn định. Batch gây OOM thì hàm tự lùi về từng ảnh.

`group_duplicates` gom keyframe gần giống nhau bằng dHash **trước khi** gọi model — chỉ ảnh đại diện mới tốn GPU, ảnh trùng chép lại caption và ghi `duplicate_of`. Đây là phần tiết kiệm thời gian lớn nhất, và chạy trên CPU nên gần như miễn phí.

In [ ]:
import json
import re
from PIL import Image, ImageOps

def load_image(image_path):
    image = ImageOps.exif_transpose(Image.open(image_path)).convert('RGB')
    if MAX_IMAGE_SIDE and max(image.size) > MAX_IMAGE_SIDE:
        image.thumbnail((MAX_IMAGE_SIDE, MAX_IMAGE_SIDE), Image.Resampling.LANCZOS)
    return image

def build_prompt_text():
    messages = [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': PROMPT}]}]
    return processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

PROMPT_TEXT = build_prompt_text()

# Mở đầu kiểu "The image shows..." không mang thông tin nhưng chiếm chỗ trong
# embedding và làm mọi caption trông giống nhau -> cắt bỏ. Model vẫn sinh ra dù
# prompt đã cấm, nên phải lọc ở đây.
META_PREFIXES = (
    'the image shows', 'the image depicts', 'the image features', 'the image captures',
    'this image shows', 'this image depicts', 'the photo shows', 'the picture shows',
    'the video keyframe shows', 'this keyframe shows', 'the keyframe shows',
    'the scene shows', 'the frame shows', 'in this image,', 'in this keyframe,',
    'in the image,', 'here we see', 'we see', 'it shows',
)

SENTENCE_SPLIT = re.compile(r'(?<=[.!?])\s+')

def drop_incomplete_sentence(text):
    """Bỏ câu cuối bị cắt dở khi chạm trần token. KHÔNG giới hạn số câu —
    độ dài là do prompt quyết định, đây chỉ dọn mảnh vụn cơ học."""
    if not text:
        return text
    parts = SENTENCE_SPLIT.split(text)
    if len(parts) > 1 and not parts[-1].rstrip().endswith(('.', '!', '?')):
        parts = parts[:-1]
    return ' '.join(parts).strip()

def clean_caption(text):
    text = (text or '').strip()
    for prefix in ('assistant\n', 'Assistant:', 'assistant:'):
        if text.startswith(prefix):
            text = text[len(prefix):].strip()
    lowered = text.lower()
    for prefix in META_PREFIXES:
        if lowered.startswith(prefix):
            text = text[len(prefix):].lstrip(' ,:')
            text = text[:1].upper() + text[1:]  # viết hoa lại sau khi cắt
            break
    return drop_incomplete_sentence(' '.join(text.split()))

# Đếm số lần model chạm trần MAX_NEW_TOKENS (bị cắt giữa chừng) để biết trần
# đang đặt đủ chưa. Caption bị cắt vẫn dùng được nhờ drop_incomplete_sentence,
# nhưng tỉ lệ cao nghĩa là đang mất thông tin -> nên tăng MAX_NEW_TOKENS.
TRUNCATION_STATS = {'total': 0, 'truncated': 0}

@torch.inference_mode()
def caption_images(images):
    """Sinh caption cho một batch ảnh PIL, trả về list[str] cùng thứ tự."""
    if not images:
        return []
    inputs = processor(text=[PROMPT_TEXT] * len(images), images=list(images),
                       return_tensors='pt', padding=True)
    inputs = {k: (v.to(model.device) if hasattr(v, 'to') else v) for k, v in inputs.items()}
    generated = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    trimmed = generated[:, inputs['input_ids'].shape[1]:]
    # Chuỗi sinh xong bình thường luôn chứa EOS; chuỗi chạm trần thì không có.
    eos_id = tokenizer.eos_token_id
    for row in trimmed:
        TRUNCATION_STATS['total'] += 1
        TRUNCATION_STATS['truncated'] += int(eos_id not in row.tolist())
    return [clean_caption(t) for t in processor.batch_decode(trimmed, skip_special_tokens=True)]

def truncation_rate():
    total = TRUNCATION_STATS['total']
    return 0.0 if not total else TRUNCATION_STATS['truncated'] / total

def frame_hash(image_path):
    """dHash 64-bit. draft() cho JPEG decode ở độ phân giải thấp -> nhanh hơn nhiều lần."""
    with Image.open(image_path) as image:
        image.draft('L', (DEDUP_HASH_SIZE * 4, DEDUP_HASH_SIZE * 4))
        small = image.convert('L').resize((DEDUP_HASH_SIZE + 1, DEDUP_HASH_SIZE),
                                          Image.Resampling.BILINEAR)
    pixels = list(small.getdata())
    bits = 0
    for row in range(DEDUP_HASH_SIZE):
        base = row * (DEDUP_HASH_SIZE + 1)
        for col in range(DEDUP_HASH_SIZE):
            bits = (bits << 1) | int(pixels[base + col] > pixels[base + col + 1])
    return bits

def group_duplicates(image_paths):
    """[(ảnh đại diện, [ảnh trùng...]), ...] — chỉ ảnh đại diện mới đưa vào model."""
    if DEDUP_MAX_DISTANCE < 0:
        return [(path, []) for path in image_paths]
    groups, reference = [], None
    for path in image_paths:
        digest = frame_hash(path)
        # So với ảnh ĐẠI DIỆN chứ không phải ảnh liền trước: tránh cảnh pan chậm
        # trôi dần từng chút một rồi gộp nhầm cả đoạn dài thành một nhóm.
        if groups and bin(digest ^ reference).count('1') <= DEDUP_MAX_DISTANCE:
            groups[-1][1].append(path)
        else:
            groups.append((path, []))
            reference = digest
    return groups

def caption_paths(image_paths):
    captions = []
    for start in range(0, len(image_paths), BATCH_SIZE):
        batch = image_paths[start:start + BATCH_SIZE]
        images = [load_image(p) for p in batch]
        try:
            captions.extend(caption_images(images))
        except torch.cuda.OutOfMemoryError:
            # Rơi về từng ảnh một thay vì hỏng cả video.
            torch.cuda.empty_cache()
            print('      OOM ở batch, chuyển sang từng ảnh (cân nhắc giảm BATCH_SIZE)')
            for image in images:
                captions.extend(caption_images([image]))
    return captions

## Thử vài ảnh trước

Lấy `TEST_COUNT` ảnh rải đều một video, in caption, đo giây/ảnh **và tỉ lệ trùng** để ước lượng thời gian thật. Cell này **không ghi gì vào Drive**.

Xem tỉ lệ trùng in ra rồi cân nhắc chỉnh `DEDUP_MAX_DISTANCE`: caption ở cell preview bên dưới mà thấy các ảnh khác cảnh bị gộp chung thì hạ xuống 2, còn thấy nhiều ảnh y hệt vẫn bị caption riêng thì nâng lên 6.

In [ ]:
import time

TEST_COUNT = 6  #@param {type:'integer'}
TEST_VIDEO_INDEX = 0  #@param {type:'integer'}

assert video_dirs, 'Không tìm thấy video nào'
sample_video = video_dirs[TEST_VIDEO_INDEX]
all_frames = sorted(
    (p for p in sample_video.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS),
    key=lambda p: (keyframe_order(p) is None, keyframe_order(p) or 0, p.name),
)
# Lấy rải đều cả video thay vì 6 ảnh đầu (thường là intro, không đại diện).
step = max(1, len(all_frames) // max(1, TEST_COUNT))
sample_paths = all_frames[::step][:TEST_COUNT]

started = time.time()
sample_captions = caption_paths(sample_paths)
elapsed = time.time() - started
for path, caption in zip(sample_paths, sample_captions):
    print(f'{path.name}: {caption}\n')
per_image = elapsed / max(1, len(sample_paths))
print(f'{sample_video.name}: {len(sample_paths)} ảnh trong {elapsed:.1f}s ({per_image:.2f}s/ảnh)')

rate = truncation_rate()
verdict = ('OK, trần đang đủ rộng' if rate == 0 else
           'chấp nhận được' if rate <= 0.1 else
           f'CAO — nên tăng MAX_NEW_TOKENS (đang {MAX_NEW_TOKENS})')
print(f"Chạm trần MAX_NEW_TOKENS: {TRUNCATION_STATS['truncated']}/{TRUNCATION_STATS['total']} "
      f'ảnh ({rate * 100:.0f}%) -> {verdict}\n')

hash_started = time.time()
groups = group_duplicates(all_frames)
saved = 1 - len(groups) / max(1, len(all_frames))
print(f'Lọc trùng (DEDUP_MAX_DISTANCE={DEDUP_MAX_DISTANCE}): {len(all_frames)} keyframe -> '
      f'{len(groups)} ảnh cần model, bỏ qua {saved * 100:.0f}% '
      f'(hash mất {time.time() - hash_started:.1f}s)')
print(f'Ước lượng 1 video: {len(groups) * per_image / 60:.1f} phút '
      f'(không lọc trùng: {len(all_frames) * per_image / 60:.1f} phút)')
print(f'Ước lượng {len(video_dirs)} video: '
      f'{len(video_dirs) * len(groups) * per_image / 3600:.1f} giờ')

## Xem ảnh kèm caption (kiểm tra bằng mắt)

Hiện đúng mấy ảnh vừa test ở trên. Ảnh lưu ở `/content/vlm_preview/` (bộ nhớ tạm Colab, không phải Drive).

In [ ]:
import textwrap
import matplotlib.pyplot as plt

PREVIEW_DIR = Path('/content/vlm_preview')  # bộ nhớ tạm Colab, KHÔNG ghi vào Drive
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)

for image_path, caption in zip(sample_paths, sample_captions):
    image = Image.open(image_path).convert('RGB')
    plt.figure(figsize=(11, 11 * image.height / image.width + 1.6))
    plt.imshow(image)
    plt.axis('off')
    plt.title(f'{sample_video.name} / {image_path.name}\n'
              + textwrap.fill(caption or '(rỗng)', 90), fontsize=10, loc='left')
    plt.tight_layout()
    plt.savefig(PREVIEW_DIR / f'{sample_video.name}_{image_path.stem}.jpg', dpi=90)
    plt.show()

print('Ảnh preview:', PREVIEW_DIR)

## Chạy theo video, có checkpoint

Checkpoint ghi ra `/content/vlm_checkpoint/` sau **mỗi batch** (đĩa local, nhanh) và đẩy lên Drive mỗi `DRIVE_SYNC_EVERY` batch (Drive I/O chậm, ghi mỗi batch sẽ ngốn đáng kể thời gian ở video dài). Mất session thì bản trên Drive vẫn cứu được phần lớn; notebook tự lấy bản dài hơn giữa local và Drive.

Xong video thì ghi `.json` hoàn chỉnh lên Drive và xoá cả hai checkpoint. `run_id` gồm model + ngôn ngữ nên đổi model không bị nhận nhầm file cũ.

In [ ]:
LOCAL_CHECKPOINT_DIR = Path('/content/vlm_checkpoint')
LOCAL_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def output_json_path(video_id): return OUTPUT_ROOT / f'{video_id}.json'
def drive_partial_path(video_id): return OUTPUT_ROOT / f'{video_id}.partial.json'
def local_partial_path(video_id): return LOCAL_CHECKPOINT_DIR / f'{video_id}.partial.json'

def atomic_write(path, payload):
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    temp.replace(path)

def load_partial(video_id):
    """Bản local là của session này, bản Drive là của session trước — lấy bản dài hơn."""
    best = None
    for path in (local_partial_path(video_id), drive_partial_path(video_id)):
        if not path.is_file(): continue
        try: payload = json.loads(path.read_text(encoding='utf-8'))
        except Exception: continue
        if payload.get('run_id') != RUN_ID: continue  # đổi model/prompt thì bỏ, làm lại
        if best is None or len(payload.get('keyframes', [])) > len(best['keyframes']):
            best = payload
    return best

def caption_video(video_dir, progress_every=100):
    video_id, mapping = video_dir.name, load_keyframe_map(video_dir.name)
    images = sorted((p for p in video_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS),
                    key=lambda p: (keyframe_order(p) is None, keyframe_order(p) or 0, p.name))
    payload = None if OVERWRITE else load_partial(video_id)
    if payload is None:
        payload = {'video_id': video_id, 'source': str(video_dir), 'model': MODEL_ID,
                   'run_id': RUN_ID, 'language': CAPTION_LANGUAGE, 'prompt': PROMPT,
                   'max_new_tokens': MAX_NEW_TOKENS, 'max_image_side': MAX_IMAGE_SIDE,
                   'dedup_max_distance': DEDUP_MAX_DISTANCE,
                   'has_keyframe_map': mapping is not None, 'complete': False,
                   'keyframe_count': 0, 'captioned_count': 0, 'keyframes': []}
    done = {item['keyframe'] for item in payload['keyframes']}
    pending = [p for p in images if p.name not in done]
    groups = group_duplicates(pending)
    if pending and len(groups) < len(pending):
        print(f'    {len(pending)} keyframe -> {len(groups)} ảnh cần model '
              f'({100 * (1 - len(groups) / len(pending)):.0f}% trùng)')

    def record(image_path, caption, duplicate_of):
        order = keyframe_order(image_path)
        mapped = mapping.get(order) if mapping and order is not None else None
        payload['keyframes'].append({
            'keyframe': image_path.name, 'n': order,
            'frame_idx': mapped['frame_idx'] if mapped else None,
            'pts_time': mapped['pts_time'] if mapped else None,
            'fps': mapped['fps'] if mapped else None,
            'caption': caption, 'duplicate_of': duplicate_of,
        })

    for batch_index, start in enumerate(range(0, len(groups), BATCH_SIZE), 1):
        chunk = groups[start:start + BATCH_SIZE]
        for (representative, duplicates), caption in zip(chunk, caption_paths([g[0] for g in chunk])):
            record(representative, caption, None)
            for duplicate in duplicates:
                record(duplicate, caption, representative.name)
        payload['keyframe_count'] = len(payload['keyframes'])
        payload['captioned_count'] = sum(k['duplicate_of'] is None for k in payload['keyframes'])
        atomic_write(local_partial_path(video_id), payload)  # đĩa local: rẻ, mỗi batch
        if DRIVE_SYNC_EVERY and batch_index % DRIVE_SYNC_EVERY == 0:
            atomic_write(drive_partial_path(video_id), payload)  # Drive: chậm, thưa hơn
        processed = min(start + BATCH_SIZE, len(groups))
        if progress_every and processed % progress_every < BATCH_SIZE:
            print(f'    {processed}/{len(groups)} ảnh')

    payload['complete'] = True
    payload['keyframes'].sort(key=lambda k: (k['n'] is None, k['n'] or 0, k['keyframe']))
    destination = output_json_path(video_id)
    atomic_write(destination, payload)
    for leftover in (local_partial_path(video_id), drive_partial_path(video_id)):
        if leftover.exists(): leftover.unlink()
    return payload, destination

## Chọn đợt chạy

In tiến độ và chốt danh sách video cho đợt này. Xem kỹ danh sách in ra rồi mới chạy cell kế tiếp.

**Chia theo folder** — cách đơn giản nhất để chia nhiều lần chạy: sửa `TARGET_FOLDERS` ở cell cấu hình, để lại vài folder mỗi lần, rồi đặt `RUN_SLICE = ''` để chạy hết phần đó. Mọi lần chạy ghi chung một thư mục Drive, mỗi video một file nên không đụng nhau.

**Chia nhỏ trong một lần** — với `SKIP_DONE = True` (mặc định), `RUN_SLICE` cắt trên **danh sách chưa xong**, nên cứ để `'0:3'` rồi chạy lại cặp cell này nhiều lần, mỗi lần tự lấy 3 video mới.

In [ ]:
def is_done(video_id):
    path = output_json_path(video_id)
    if not path.is_file(): return False
    try: payload = json.loads(path.read_text(encoding='utf-8'))
    except Exception: return False
    return payload.get('run_id') == RUN_ID and bool(payload.get('complete'))

def parse_slice(text):
    text = (text or '').strip()
    if not text: return None
    start, sep, stop = text.partition(':')
    if not sep: return slice(int(start), int(start) + 1)
    return slice(int(start) if start.strip() else None, int(stop) if stop.strip() else None)

def progress_report():
    done = [v.name for v in video_dirs if is_done(v.name)]
    pending = [v.name for v in video_dirs if v.name not in set(done)]
    partial = sorted({p.name.split('.')[0] for p in
                      list(OUTPUT_ROOT.glob('*.partial.json')) + list(LOCAL_CHECKPOINT_DIR.glob('*.partial.json'))}
                     - set(done))
    print(f'TARGET_FOLDERS lần này: {len(video_dirs)} video | đã xong {len(done)} | còn {len(pending)}'
          + (f' | đang dở: {", ".join(partial)}' if partial else ''))
    # Đếm cả file đã có trên Drive từ các lần chạy TARGET_FOLDERS khác.
    on_drive = len(list(OUTPUT_ROOT.glob('L*_V*.json')))
    print(f'Tổng cộng trên Drive (mọi folder đã chạy): {on_drive} file JSON')
    return done, pending

def resolve_batch(verbose=True):
    """Danh sách video cho ĐỢT chạy này, theo RUN_VIDEO_IDS hoặc RUN_SLICE."""
    if RUN_VIDEO_IDS:
        by_id = {v.name: v for v in video_dirs}
        unknown = [i for i in RUN_VIDEO_IDS if i not in by_id]
        assert not unknown, f'Không tìm thấy video: {unknown}'
        chosen = [by_id[i] for i in RUN_VIDEO_IDS]
        source = 'RUN_VIDEO_IDS'
    else:
        pool = [v for v in video_dirs if not is_done(v.name)] if SKIP_DONE else list(video_dirs)
        window = parse_slice(RUN_SLICE)
        chosen = pool[window] if window else pool
        source = ('danh sách chưa xong' if SKIP_DONE else 'toàn bộ danh sách') + f'[{RUN_SLICE or ":"}]'
    if verbose:
        print(f'Đợt này: {len(chosen)} video (nguồn: {source})')
        print('  ' + (', '.join(v.name for v in chosen) if chosen else '(rỗng — có thể đã xong hết)'))
    return chosen

progress_report()
run_batch = resolve_batch()

## Chạy đợt này

In [ ]:
import time
import traceback

success = skipped = failed = 0
failures = []
batch_started = time.time()
for index, video_dir in enumerate(run_batch, 1):
    if is_done(video_dir.name) and not OVERWRITE:
        skipped += 1; print(f'[{index}/{len(run_batch)}] SKIP {video_dir.name}'); continue
    print(f'[{index}/{len(run_batch)}] VLM  {video_dir.name}')
    started = time.time()
    try:
        payload, _ = caption_video(video_dir)
        filled = sum(bool(k['caption']) for k in payload['keyframes'])
        print(f"    {filled}/{payload['keyframe_count']} keyframe có caption "
              f"({payload['captioned_count']} lượt gọi model, {time.time() - started:.0f}s)")
        success += 1
    except Exception as exc:
        failed += 1; failures.append({'video_id': video_dir.name, 'error': repr(exc)}); traceback.print_exc()
    finally:
        if use_gpu: torch.cuda.empty_cache()

# Gộp vào _failed.json cũ thay vì ghi đè, để không mất lỗi của các đợt trước.
failed_path, history = OUTPUT_ROOT / '_failed.json', []
if failed_path.is_file():
    try:
        current = {v.name for v in run_batch}
        history = [row for row in json.loads(failed_path.read_text(encoding='utf-8'))
                   if row.get('video_id') not in current]
    except Exception: history = []
atomic_write(failed_path, history + failures)

print(f'\nĐợt này xong sau {(time.time() - batch_started) / 60:.1f} phút: '
      f'success={success}, skipped={skipped}, failed={failed}')
rate = truncation_rate()
print(f"Chạm trần MAX_NEW_TOKENS: {TRUNCATION_STATS['truncated']}/{TRUNCATION_STATS['total']} ({rate * 100:.0f}%)")
if rate > 0.1:
    print(f'  -> Nên tăng MAX_NEW_TOKENS (đang {MAX_NEW_TOKENS}) cho các đợt sau; '
          'caption đang bị cắt mất phần cuối.')
progress_report()

## Xuất JSONL để index

Mỗi dòng có cả `caption` lẫn `text` (cùng nội dung) để dùng chung pipeline index với OCR, kèm `duplicate_of` nếu muốn lọc bớt bản trùng khi build index.

Caption là **kênh phụ trợ cho CLIP, không thay thế CLIP**. Cách dùng: embed `caption` bằng một text encoder câu dài (BGE-M3, E5) thành FAISS index riêng, rồi hợp nhất với kênh CLIP bằng Reciprocal Rank Fusion. Thêm BM25 trên `caption` để bắt truy vấn có từ khoá hiếm. Đừng thay CLIP bằng caption — caption chỉ giữ được một phần nhỏ thông tin của ảnh.

In [ ]:
jsonl_path = OUTPUT_ROOT / 'vlm_index.jsonl'
output_lines = no_frame_idx = 0
with jsonl_path.open('w', encoding='utf-8') as handle:
    for json_path in sorted(OUTPUT_ROOT.glob('L*_V*.json')):
        payload = json.loads(json_path.read_text(encoding='utf-8'))
        if payload.get('run_id') != RUN_ID or not payload.get('complete'): continue
        for keyframe in payload['keyframes']:
            if not keyframe['caption']: continue
            no_frame_idx += keyframe['frame_idx'] is None
            row = {'video_id': payload['video_id'], 'frame_idx': keyframe['frame_idx'],
                   'pts_time': keyframe['pts_time'], 'keyframe': keyframe['keyframe'],
                   'caption': keyframe['caption'], 'text': keyframe['caption'],
                   'duplicate_of': keyframe.get('duplicate_of')}
            handle.write(json.dumps(row, ensure_ascii=False) + '\n'); output_lines += 1
print(f'Đã ghi {output_lines} dòng vào {jsonl_path}')
if no_frame_idx: print(f'CẢNH BÁO: {no_frame_idx} dòng thiếu frame_idx')